# 🛰️ HỆ THỐNG TRUYỀN DỮ LIỆU QUA VỆ TINH IRIDIUM - ROCKBLOCK 9603
## 📡 MODULE GỬI (SENDER) - SERIAL / ID: 0235707

### 📖 Giới thiệu & Nguyên lý hoạt động:
* **RockBLOCK 9603** là module thu phát vệ tinh sử dụng chùm vệ tinh tầm thấp **Iridium LEO**, cho phép gửi nhận dữ liệu toàn cầu (kể cả giữa biển hoặc vùng không có sóng 4G/GSM).
* **Cơ chế gửi tin (MO - Mobile Originated):**
  1. Giao tiếp qua UART/Serial (Baudrate mặc định: `19200`, 8N1).
  2. Nạp nội dung vào bộ đệm MO của modem bằng lệnh `AT+SBDWT=<Nội dung>`.
  3. Kích hoạt phiên truyền dữ liệu vệ tinh SBD bằng lệnh `AT+SBDIX`.
  4. Modem bắt sóng vệ tinh, truyền tin nhắn lên vệ tinh Iridium và nhận lại mã phản hồi từ Gateway.
  5. Gateway chuyển tiếp tin nhắn tới RockBLOCK nhận (0235708) hoặc Cloud Webhook.
  6. Xóa bộ đệm MO bằng lệnh `AT+SBDD0` để hoàn tất phiên gửi.


In [ ]:
# CELL 1: KHAI BÁO THƯ VIỆN VÀ CẤU HÌNH THIẾT BỊ
import time
import serial
import serial.tools.list_ports
from datetime import datetime

# ==============================================================================
# DANH SÁCH CỔNG SERIAL HIỆN TẠI TRÊN MÁY TÍNH
# ==============================================================================
print("🔍 Đang quét các cổng COM khả dụng trên máy:")
ports = list(serial.tools.list_ports.comports())
if not ports:
    print("  [!] Không tìm thấy cổng COM nào. Hãy kiểm tra kết nối cáp USB-FTDI!")
else:
    for p in ports:
        print(f"  -> Cổng: {p.device:<8} | Mô tả: {p.description}")

# ==============================================================================
# CẤU HÌNH KẾT NỐI CHO THIẾT BỊ GỬI (0235707)
# (Hãy điều chỉnh PORT tương ứng với cổng cắm RockBLOCK 0235707 của bạn)
# ==============================================================================
PORT = "COM14"          # Ví dụ: COM14 (hoặc COM3 nếu test mô phỏng)
BAUDRATE = 19200        # Tốc độ baud chuẩn của RockBLOCK 9603 là 19200
TIMEOUT = 3             # Thời gian timeout chờ đọc serial (giây)
LOG_FILE = "sender_0235707_log.txt"

print(f"\n⚙️ Cấu hình thiết bị gửi: Port = {PORT}, Baud = {BAUDRATE}, Log = {LOG_FILE}")


In [ ]:
# CELL 2: ĐỊNH NGHĨA CLASS ĐIỀU KHIỂN ROCKBLOCK SENDER
class RockBlockSender:
    """Lớp điều khiển module RockBLOCK phát tín hiệu SBD qua mạng Iridium."""
    
    # Bảng giải thích mã kết quả MO Status từ lệnh AT+SBDIX
    MO_STATUS_DESC = {
        0: "Thành công: Tin nhắn MO đã được truyền lên Gateway vệ tinh thành công.",
        1: "Thành công: Tin nhắn MO đã truyền, nhưng kích thước tin nhắn quá lớn.",
        2: "Thành công: Tin nhắn MO đã truyền, nhưng dữ liệu vị trí không cập nhật được.",
        32: "Thất bại: Không kết nối được mạng vệ tinh Iridium (Hết timeout / Anten bị che khuất)."
    }

    def __init__(self, port, baudrate=19200, timeout=3, log_file="sender_log.txt"):
        self.port = port
        self.baudrate = baudrate
        self.timeout = timeout
        self.log_file = log_file
        self.ser = None
        self.imei = None

    def log(self, message):
        """In log ra console và lưu vào file log với timestamp."""
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_entry = f"[{now}] {message}"
        print(log_entry)
        with open(self.log_file, "a", encoding="utf-8") as f:
            f.write(log_entry + "\n")

    def connect(self):
        """Mở cổng Serial kết nối tới RockBLOCK."""
        try:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=self.baudrate,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_NONE,
                stopbits=serial.STOPBITS_ONE,
                timeout=self.timeout
            )
            self.log(f"✅ Đã kết nối thành công tới cổng {self.port} @ {self.baudrate} baud.")
            return True
        except serial.SerialException as e:
            self.log(f"❌ Lỗi mở cổng Serial: {e}")
            return False

    def disconnect(self):
        """Đóng kết nối Serial an toàn."""
        if self.ser and self.ser.is_open:
            self.ser.close()
            self.log("🔌 Đã ngắt kết nối cổng Serial.")

    def send_cmd(self, cmd, wait=1.5):
        """Gửi lệnh AT tới modem và đọc phản hồi."""
        if not self.ser or not self.ser.is_open:
            self.log("❌ Cổng Serial chưa mở!")
            return ""
        
        self.ser.reset_input_buffer()
        self.ser.write((cmd + "\r").encode("ascii"))
        time.sleep(wait)
        
        response = self.ser.read_all().decode("ascii", errors="ignore").strip()
        return response

    def init_modem(self):
        """Khởi tạo cấu hình ban đầu cho RockBLOCK."""
        self.log("--- Bắt đầu khởi tạo cấu hình modem ---")
        
        # 1. Kiểm tra phản hồi cơ bản
        resp = self.send_cmd("AT", wait=1)
        if "OK" not in resp:
            self.log(f"⚠️ Modem không phản hồi lệnh AT: {resp}")
            return False
            
        # 2. Tắt echo (ATE0) để phản hồi sạch sẽ
        self.send_cmd("ATE0", wait=1)
        
        # 3. Tắt phần cứng flow control (AT&K0)
        self.send_cmd("AT&K0", wait=1)
        
        # 4. Lấy thông tin phiên bản firmware và IMEI
        fw = self.send_cmd("AT+CGMR", wait=1.5)
        self.imei = self.send_cmd("AT+CGSN", wait=1.5).replace("OK", "").strip()
        
        self.log(f"📱 IMEI thiết bị : {self.imei}")
        self.log(f"💾 Firmware      : {fw.replace('OK', '').strip()}")
        self.log("✅ Modem đã sẵn sàng hoạt động!")
        return True

    def check_signal(self):
        """Kiểm tra mức sóng vệ tinh Iridium (AT+CSQ). Trả về số nguyên từ 0 đến 5."""
        resp = self.send_cmd("AT+CSQ", wait=2)
        # Phản hồi thường có định dạng: +CSQ:X
        if "+CSQ:" in resp:
            try:
                bars_str = resp.split("+CSQ:")[1].strip().split()[0]
                bars = int(bars_str)
                bar_display = "■" * bars + "□" * (5 - bars)
                self.log(f"📶 Mức tín hiệu vệ tinh: [{bar_display}] ({bars}/5 vạch)")
                return bars
            except Exception:
                pass
        self.log(f"⚠️ Không đọc được mức sóng. Phản hồi: {resp}")
        return 0

    def wait_for_signal(self, min_bars=2, max_attempts=10, interval=5):
        """Vòng lặp chờ sóng vệ tinh đạt mức tối thiểu trước khi gửi."""
        self.log(f"⏳ Đang chờ sóng vệ tinh đạt tối thiểu {min_bars}/5 vạch...")
        for i in range(1, max_attempts + 1):
            bars = self.check_signal()
            if bars >= min_bars:
                self.log(f"🎉 Đã bắt được sóng vệ tinh đủ mạnh ({bars}/5 vạch)!")
                return True
            if i < max_attempts:
                self.log(f"   [Lần {i}/{max_attempts}] Sóng chưa đủ. Đợi {interval}s (Lưu ý: Hướng anten thẳng lên trời)...")
                time.sleep(interval)
        self.log(f"❌ Sau {max_attempts} lần kiểm tra vẫn chưa có đủ sóng vệ tinh.")
        return False

    def send_message(self, message_text, wait_sbd=20):
        """
        Nạp tin nhắn và truyền qua vệ tinh Iridium.
        1. AT+SBDWT: Ghi tin nhắn vào bộ đệm MO.
        2. AT+SBDIX: Khởi tạo phiên truyền dữ liệu vệ tinh SBD.
        3. Phân tích kết quả +SBDIX: <mo_stat>, <momsn>, <mt_stat>, <mtmsn>, <mt_len>, <mt_queued>
        4. AT+SBDD0: Dọn dẹp bộ đệm MO.
        """
        self.log("=" * 60)
        self.log(f"📤 BẮT ĐẦU QUÁ TRÌNH GỬI THÔNG ĐIỆP:")
        self.log(f"📝 Nội dung: \"{message_text}\"")
        self.log(f"📏 Độ dài  : {len(message_text.encode('utf-8'))} bytes")
        self.log("=" * 60)

        # Bước 1: Xóa bộ đệm MO cũ để tránh gửi sót dữ liệu trước
        self.send_cmd("AT+SBDD0", wait=1)

        # Bước 2: Nạp tin nhắn văn bản vào MO Buffer
        write_cmd = f"AT+SBDWT={message_text}"
        write_resp = self.send_cmd(write_cmd, wait=1)
        if "OK" not in write_resp:
            self.log(f"❌ Không thể nạp tin nhắn vào bộ đệm! Phản hồi: {write_resp}")
            return False
        self.log("✅ Đã nạp thông điệp vào bộ đệm MO.")

        # Bước 3: Kích hoạt phiên truyền dữ liệu vệ tinh SBD (AT+SBDIX)
        self.log("🚀 Đang gọi AT+SBDIX (Phiên truyền vệ tinh, vui lòng đợi 10-25 giây)...")
        # Phiên SBDIX có thể mất từ 10 đến 25 giây để hoàn tất bắt tay vệ tinh
        sbdix_resp = self.send_cmd("AT+SBDIX", wait=wait_sbd)
        self.log(f"📥 Phản hồi từ phiên SBDIX:\n{sbdix_resp}")

        # Bước 4: Phân tích kết quả trả về
        if "+SBDIX:" in sbdix_resp:
            try:
                # Định dạng: +SBDIX: <mo_stat>, <momsn>, <mt_stat>, <mtmsn>, <mt_len>, <mt_queued>
                raw_params = sbdix_resp.split("+SBDIX:")[1].strip().split(",")
                mo_status = int(raw_params[0].strip())
                momsn = int(raw_params[1].strip())
                mt_status = int(raw_params[2].strip())
                mt_len = int(raw_params[4].strip())
                mt_queued = int(raw_params[5].strip())

                desc = self.MO_STATUS_DESC.get(mo_status, f"Mã trạng thái MO không xác định: {mo_status}")
                self.log(f"📊 Kết quả MO Status = {mo_status}: {desc}")
                self.log(f"🔢 MOMSN (Số thứ tự tin nhắn gửi): {momsn}")
                
                # Kiểm tra nếu đồng thời có tin nhắn MT gửi về
                if mt_status == 1:
                    self.log(f"📬 [Thông báo kèm theo] Modem nhận được 1 tin nhắn MT ({mt_len} bytes) từ Gateway!")

                # Bước 5: Xóa bộ đệm MO sau khi gửi
                self.send_cmd("AT+SBDD0", wait=1)

                if mo_status in [0, 1, 2]:
                    self.log("✨ KẾT QUẢ: GỬI TIN NHẮN QUA VỆ TINH THÀNH CÔNG RỰC RỠ!")
                    return True
                else:
                    self.log(f"⚠️ GỬI THẤT BẠI: Mã lỗi MO = {mo_status}. (Gợi ý: Cần anten ngoài trời, góc quét bầu trời thoáng)")
                    return False
            except Exception as ex:
                self.log(f"❌ Lỗi khi phân tích chuỗi phản hồi SBDIX: {ex}")
                return False
        else:
            self.log("❌ Modem không trả về chuỗi kết quả +SBDIX. Vui lòng kiểm tra lại cáp và nguồn cấp!")
            return False


In [ ]:
# CELL 3: KHỞI TẠO ĐỐI TƯỢNG VÀ KẾT NỐI VỚI ROCKBLOCK SENDER
sender = RockBlockSender(port=PORT, baudrate=BAUDRATE, log_file=LOG_FILE)

# Mở cổng kết nối
if sender.connect():
    # Cấu hình modem và đọc IMEI
    ready = sender.init_modem()
    if not ready:
        print("⚠️ Khởi tạo modem thất bại. Vui lòng kiểm tra cổng COM hoặc rút cắm lại thiết bị.")
else:
    print("❌ Không mở được cổng Serial. Hãy kiểm tra lại biến PORT ở Cell 1!")


In [ ]:
# CELL 4: KIỂM TRA SÓNG VỆ TINH TRƯỚC KHI GỬI
# Lưu ý: Mạng Iridium hoạt động tốt nhất khi anten được đặt ngoài trời hoặc sát cửa sổ hướng lên trời.
print("📡 Bắt đầu kiểm tra sóng Iridium...")
has_signal = sender.wait_for_signal(min_bars=2, max_attempts=6, interval=5)

if has_signal:
    print("✅ Sóng vệ tinh sẵn sàng cho việc phát tin.")
else:
    print("⚠️ Cảnh báo: Sóng vệ tinh yếu hoặc chưa bắt được. Nếu gửi có thể gặp lỗi MO=32 (Timeout).")


In [ ]:
# CELL 5: THỰC HIỆN GỬI THÔNG ĐIỆP ĐƠN (TEST MESSAGE)
# Chuỗi tin nhắn gửi sang thiết bị 0235708
# Bạn có thể thay đổi nội dung tin nhắn tùy ý bên dưới:
custom_message = f"HELLO FROM 0235707 TO 0235708 TIME={datetime.now().strftime('%H:%M:%S')}"

# Thực hiện truyền dữ liệu
success = sender.send_message(custom_message, wait_sbd=20)

if success:
    print(f"🎉 Tin nhắn '{custom_message}' đã được gửi lên mạng Iridium thành công!")
else:
    print("❌ Gửi tin nhắn thất bại. Hãy kiểm tra anten và góc nhìn bầu trời.")


In [ ]:
# CELL 6: VÍ DỤ GỬI DỮ LIỆU ĐỊNH VỊ HOẶC CẢM BIẾN (TELEMETRY)
# Định dạng dữ liệu thu gọn tiết kiệm băng thông SBD
telemetry_data = {
    "dev": "0235707",
    "lat": 21.0285,
    "lon": 105.8542,
    "temp": 28.5,
    "bat": 98
}

# Chuyển đổi thành chuỗi ngắn gọn dạng CSV để tiết kiệm byte truyền qua vệ tinh
compact_msg = f"T:{telemetry_data['dev']},{telemetry_data['lat']},{telemetry_data['lon']},{telemetry_data['temp']},{telemetry_data['bat']}"

print(f"📦 Dữ liệu nén gọn ({len(compact_msg)} bytes): {compact_msg}")

# Bỏ chú thích dòng dưới để gửi dữ liệu telemetry khi muốn:
# sender.send_message(compact_msg, wait_sbd=20)


In [ ]:
# CELL 7: ĐÓNG KẾT NỐI KHI HOÀN TẤT
sender.disconnect()
print(f"📄 Toàn bộ tiến trình đã được lưu lại tại file: {LOG_FILE}")
